In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [15]:
df = pd.read_csv('../data/bank/bank-full.csv', sep=';')


In [16]:
X = df.drop('y', axis=1)
y = df['y']

In [17]:
import sys

if '..' not in sys.path:
    sys.path.append('..')

from src import model_config



In [18]:
from sklearn.model_selection import train_test_split

X = df.drop('y', axis=1)
y = df['y']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [29]:
# Convert target variables from strings to integers
y_train = y_train.map({'yes': 1, 'no': 0})
y_test = y_test.map({'yes': 1, 'no': 0})

In [30]:
X_train.shape

(36168, 16)

In [31]:
X_test.shape

(9043, 16)

In [32]:
y_train.shape

(36168,)

In [33]:
y_test.shape

(9043,)

In [34]:
print("Training set proportions:\n", y_train.value_counts(normalize=True))
print("\nTest set proportions:\n", y_test.value_counts(normalize=True))

Training set proportions:
 y
0    0.883018
1    0.116982
Name: proportion, dtype: float64

Test set proportions:
 y
0    0.883003
1    0.116997
Name: proportion, dtype: float64


In [35]:
from src.model_config import get_model_pipelines

pipelines = get_model_pipelines(X_train)

In [25]:
len(pipelines)

10

In [26]:
pipelines.keys()

dict_keys(['decision_tree_normal', 'decision_tree_balanced', 'logistic_regression_normal', 'logistic_regression_balanced', 'random_forest_normal', 'random_forest_balanced', 'svm_normal', 'svm_balanced', 'voting_ensemble_normal', 'voting_ensemble_balanced'])

In [36]:
from sklearn.model_selection import StratifiedKFold, cross_validate
import pandas as pd
import numpy as np

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {'f1': 'f1', 'pr_auc': 'average_precision'}

results = []

for name, pipeline in pipelines.items():
    
    cv_scores = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1, error_score='raise')
    
    mean_f1 = np.mean(cv_scores['test_f1'])
    std_f1 = np.std(cv_scores['test_f1'])
    mean_pr_auc = np.mean(cv_scores['test_pr_auc'])
    
    results.append({
        'Pipeline Name': name,
        'Mean CV F1': mean_f1,
        'Std CV F1': std_f1,
        'Mean CV PR-AUC': mean_pr_auc
    })

results_df = pd.DataFrame(results)

results_df.sort_values(by='Mean CV F1', ascending=False, inplace=True)
results_df.reset_index(drop=True, inplace=True)


results_df

,Pipeline Name,Mean CV F1,Std CV F1,Mean CV PR-AUC
0,random_forest_balanced,0.427883,0.014986,0.415484
1,voting_ensemble_balanced,0.393223,0.008507,0.421876
2,logistic_regression_balanced,0.372822,0.003512,0.398151
3,random_forest_normal,0.318793,0.016373,0.422028
4,decision_tree_balanced,0.304675,0.007322,0.173935
5,decision_tree_normal,0.299087,0.021897,0.170252
6,logistic_regression_normal,0.274874,0.012208,0.401805
7,svm_balanced,0.271372,0.012426,0.252292
8,voting_ensemble_normal,0.180804,0.048231,0.432346
9,svm_normal,0.000000,0.000000,0.302847


In [28]:
print(results_df)

                  Pipeline Name  Mean CV F1  Std CV F1  Mean CV PR-AUC
0          decision_tree_normal         NaN        NaN             NaN
1        decision_tree_balanced         NaN        NaN             NaN
2    logistic_regression_normal         NaN        NaN             NaN
3  logistic_regression_balanced         NaN        NaN             NaN
4          random_forest_normal         NaN        NaN             NaN
5        random_forest_balanced         NaN        NaN             NaN
6                    svm_normal         NaN        NaN             NaN
7                  svm_balanced         NaN        NaN             NaN
8        voting_ensemble_normal         NaN        NaN             NaN
9      voting_ensemble_balanced         NaN        NaN             NaN
